# [Experiment] Vanilla TabNet | Step LR | Poker Hand

### Overview
This notebook establishes the fundamental performance baseline using the standard, unmodified TabNet architecture. The hyperparameters and structural configuration of this baseline have been meticulously aligned to follow the experimental setup detailed in the original TabNet paper as exactly as possible.

### Methodological Context: The Optimization Environment
To create a rigorously grounded control, this baseline employs a standard StepLR learning rate schedule. This specific schedule was selected because it mirrors the exact optimization strategy utilized by the original authors. This discrete, step-wise decay provides a historically validated, rigid optimization trajectory against which all of our subsequent architectural modifications will be measured.

### The Objective
The primary goal of this notebook is to capture the default performance metrics—specifically convergence rate, peak accuracy, and late-stage training stability—of the standard model as originally intended by its creators. This establishes the strict "ground truth" necessary to empirically evaluate the impact of introducing Kolmogorov-Arnold Network (KAN) layers.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-training-true.data'
url_test = 'https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-testing.data'

df_train = pd.read_csv(url_train, header=None)
df_test = pd.read_csv(url_test, header=None)

# 0-index the categorical features for PyTorch embedding layers
X_train_full = df_train.iloc[:, :-1].values - 1
y_train_full = df_train.iloc[:, -1].values.astype(int)

X_test = df_test.iloc[:, :-1].values - 1
y_test = df_test.iloc[:, -1].values.astype(int)

# remove the permutation invariance trap that cripples standard MLPs on this dataset
def sort_poker_hands(X):
    # reshape from (N, 10) to (N, 5, 2) to lock each Suit with its Rank
    X_pairs = X.reshape(-1, 5, 2)

    # get the indices that sort each hand based on the Rank (index 1 of the pair)
    sort_indices = np.argsort(X_pairs[:, :, 1], axis=1)

    # apply the sort to both Suit and Rank simultaneously
    row_indices = np.arange(X.shape[0])[:, np.newaxis]
    X_sorted_pairs = X_pairs[row_indices, sort_indices]

    # flatten back to the standard (N, 10) TabNet input format
    return X_sorted_pairs.reshape(-1, 10)

print("Sorting hands to remove permutation invariance...")
X_train_full = sort_poker_hands(X_train_full)
X_test = sort_poker_hands(X_test)
print("Sorting complete.")

del df_train, df_test
gc.collect()

# split 6k samples for validation from the 25k training dataset
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=6000, random_state=seed, stratify=y_train_full
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Sorting hands to remove permutation invariance...
Sorting complete.
Train shape: (19010, 10)
Valid shape: (6000, 10)
Test shape:  (1000000, 10)


### Model Training

In [5]:
MAX_EPOCHS = 10000

In [6]:
# define categorical structures for the 10 input columns
categorical_indices = list(range(10))
categorical_dimensions = [4, 13, 4, 13, 4, 13, 4, 13, 4, 13]
categorical_embedding_dims = [1] * 10

# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.95 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 100 epochs approximates the paper's 500 iterations
#   (19k train rows / 4096 batch size = ~4.64 iters/epoch. 500 / 4.64 = ~107. Using 100).
paper_config = {
    'n_d': 16,
    'n_a': 16,
    'n_steps': 4,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.05,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': dict(lr=0.01),
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=100, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_vanilla = TabNetClassifier(
    **paper_config,
    cat_idxs=categorical_indices,
    cat_dims=categorical_dimensions,
    cat_emb_dim=categorical_embedding_dims,
    clip_value=2.,
    use_kan=False,
    seed=seed,
    verbose=500
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 4096,
    'virtual_batch_size': 1024,
}

clf_vanilla.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=MAX_EPOCHS,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 4.05375 | valid_accuracy: 0.18533 |  0:00:01s
epoch 500| loss: 0.17174 | valid_accuracy: 0.94717 |  0:02:47s
epoch 1000| loss: 0.06491 | valid_accuracy: 0.973   |  0:05:30s
epoch 1500| loss: 0.03035 | valid_accuracy: 0.9825  |  0:08:14s
epoch 2000| loss: 0.02785 | valid_accuracy: 0.9825  |  0:10:59s
epoch 2500| loss: 0.01622 | valid_accuracy: 0.98233 |  0:13:42s
epoch 3000| loss: 0.01296 | valid_accuracy: 0.98217 |  0:16:27s
epoch 3500| loss: 0.01019 | valid_accuracy: 0.98267 |  0:19:11s
epoch 4000| loss: 0.0095  | valid_accuracy: 0.984   |  0:21:55s
epoch 4500| loss: 0.0089  | valid_accuracy: 0.98233 |  0:24:39s
epoch 5000| loss: 0.00777 | valid_accuracy: 0.98033 |  0:27:22s
epoch 5500| loss: 0.00607 | valid_accuracy: 0.97967 |  0:30:07s
epoch 6000| loss: 0.00598 | valid_accuracy: 0.98083 |  0:32:51s
epoch 6500| loss: 0.00614 | valid_accuracy: 0.97967 |  0:35:34s
epoch 7000| loss: 0.00453 | valid_accuracy: 0.97967 |  0:38:19s
epoch 7500| loss: 0.00469 | valid_accuracy

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_vanilla.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 26,733


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_vanilla.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.98450


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/poker_hand/tables'
MODELS_DIR = f'{BASE_DIR}/results/poker_hand/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "Vanilla TabNet",
    "dataset": "Poker Hand",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "training_history": clf_vanilla.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/01_vanilla_baseline_step_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_vanilla.save_model(f'{MODELS_DIR}/01_vanilla_baseline_step_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/poker_hand/tables/01_vanilla_baseline_step_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/poker_hand/models/01_vanilla_baseline_step_lr_26k.zip
